
### Messages
Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

Role - Identifies the message type (e.g. system, user)
Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
Metadata - Optional fields such as response information, message IDs, and token usage
LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

## Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

## Use text prompts when:

You have a single, standalone request
You don’t need conversation history
You want minimal code complexity

## Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types

System message - Tells the model how to behave and provide context for interactions
Human message - Represents user input and interactions with the model
AI message - Responses generated by the model, including text content, tool calls, and metadata
Tool message - Represents the outputs of tool calls

# System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.

#Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

# AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

# Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")  
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool

model=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
from langchain.messages import AIMessage, HumanMessage, SystemMessage
messages=[
    SystemMessage("You are a teacher that explains AI to students in a simple way."),
    HumanMessage(content="what is AI? ")
]

response=model.invoke(messages)
response.content[0]["text"]

'Hello there! Imagine you have a really smart robotic puppy. \n\nIf you throw a ball, the puppy chases it and brings it back. That’s cool, but it’s just following simple rules it was built with. \n\nNow, imagine that same robotic puppy can learn your favorite games just by watching you play, can recognize your voice, and even knows when you are sad so it can bring you a toy. *That* is **AI**, which stands for **Artificial Intelligence**.\n\nIn simple terms:\n*   **Artificial** means it\'s made by humans, not born naturally like a real dog or a human.\n*   **Intelligence** means the ability to learn, think, and solve problems.\n\nSo, AI is basically **giving computers and machines the ability to think and learn like humans do.**\n\nYou probably use AI every day without even knowing it! \n*   When Netflix or YouTube recommends a cool video you might like, that\'s AI. \n*   When you play a video game and the computer-controlled characters try to outsmart you, that\'s AI.\n*   When you ask

In [10]:
from langchain.tools import tool

model=ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

@tool
def getWeather(city:str)->str:
    """
    Get the current weather for a given city.
    """
    return f"The current weather in {city} is sunny with a temperature of 25°C."

model_with_tools=model.bind_tools([getWeather])

messages=[
    SystemMessage("You are a assistant that can provide weather information for cities."),
    HumanMessage(content="what is the weather in Paris? ")
]

ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)
for tool_call in ai_msg.tool_calls:
    tool_result=getWeather.invoke(tool_call)
    messages.append(tool_result)

final_result=model_with_tools.invoke(messages)
final_result.text

'The current weather in Paris is sunny with a temperature of 25°C.'